# PSF simulator and PSF-aware SNR

A tour of the detector-grid **PSF simulator** and the **PSF-aware SNR**:

- **PSF sources** — `AiryPSF` (diffraction limited), `DefocusPSF` (+1 / +2 waves, from
  Zemax Huygens data), and `CustomPSF` (any image).
- **`ImageSimulator`** — `render_psf` gives the bare normalized PSF on the detector grid;
  `simulate` adds source flux, sky, dark, Poisson + read noise, and a saturation mask.
- **`Simulation.get_image_snr`** — aperture SNR from a *rendered* PSF, so it works for any
  PSF rather than only the analytic Airy disk. For defocus it quantifies the SNR penalty
  and finds the optimal aperture.

Everything is in electrons.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

import wcc_etc

wcc_etc.set_wcc_style()

scene = wcc_etc.get_scene(
    name="G5V",
    mag=18,
    background="zodi",
    bandpass="johnson_r",
    background_prop={"bandpass": "johnson_r", "mag": 22.5},
)
sim = wcc_etc.Simulation.from_sensor_and_scene("sony:r", scene)
imsim = wcc_etc.ImageSimulator.from_sensor_and_scene("sony:r", scene, npix=300)

PSFS = {
    "Airy (diffraction limited)": wcc_etc.AiryPSF(),
    "+1 wave defocus": wcc_etc.DefocusPSF(wcc_etc.DEFOCUS_1WAVE_PATH),
    "+2 wave defocus": wcc_etc.DefocusPSF(wcc_etc.DEFOCUS_2WAVE_PATH),
}

print(f"plate scale : {imsim.plate_scale_mas:.1f} mas/pix")
print(f"aperture    : {sim.meta['r_aper_mas']} mas")

## 1. PSF gallery

`ImageSimulator.render_psf` renders a PSF onto the detector grid normalized to unit sum —
no source flux, no background, no noise. Passing `jitter_sigma_mas=0` switches off the
pointing blur so the diffraction limit is visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (name, psf) in zip(axes, PSFS.items()):
    img = imsim.render_psf(psf, jitter_sigma_mas=0.0, npix=128)
    im = ax.imshow(
        img,
        origin="lower",
        norm=LogNorm(vmin=img.max() * 1e-4, vmax=img.max()),
        cmap="viridis",
    )
    ax.set_title(f"{name}\npeak fraction = {img.max():.3f}")
    ax.set_xlabel("pixel")
    ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("Normalized PSFs on the Sony grid (log stretch, no jitter)", y=1.03)
fig.tight_layout()
plt.show()

## 2. A full detector image

`simulate(time)` distributes the source electrons over the PSF, adds the per-pixel sky and
dark current, then Poisson + Gaussian read noise. The returned `SimulatedImage` carries
`image_clean`, `image_e`, a `saturation_mask`, and the `to_adu()` / `to_fitsimg()`
converters.

In [ ]:
res = imsim.simulate(time=30, seed=0)

fig, axes = res.plot_image_row(stretch="log", units="mas")
fig.suptitle("G5V r=18, 30 s, sony:r, Airy PSF", y=1.02)
fig.tight_layout()
plt.show()

print(f"clean total electrons : {res.image_clean.sum():.3e}")
print(f"peak pixel (e-)       : {res.image_e.max():.1f}")
print(f"saturated pixels      : {res.saturation_mask.sum()}")

### Radial profile and encircled energy

The same `SimulatedImage` plots its own azimuthally-averaged radial profile and
encircled-energy curve. `plot_radial` marks the HWHM; `plot_encircled_energy` marks a
target EE fraction.

In [ ]:
fig, (ax_r, ax_ee) = plt.subplots(1, 2, figsize=(11, 4))
res.plot_radial(ax=ax_r, units="mas", title="Radial profile")
res.plot_encircled_energy(
    ax=ax_ee, units="mas", ee_target=0.8, title="Encircled energy"
)
fig.tight_layout()
plt.show()

### Comparing focus levels

`render_psf` plus `plate_scale_mas` feeds the same plotting helpers directly, so several
normalized PSFs overlay on one axis. Defocus spreads the light into a broad doughnut: the
radial peak drops by orders of magnitude, and the encircled energy needs a much larger
radius to reach 80%.

In [ ]:
fig, (ax_r, ax_ee) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, psf in PSFS.items():
    psf_img = imsim.render_psf(psf, jitter_sigma_mas=0.0)
    wcc_etc.plot_radial_mpl(
        image_clean=psf_img,
        pixel_scale_mas=imsim.plate_scale_mas,
        ax=ax_r,
        units="mas",
        show_hwhm=False,
        label=name,
    )
    wcc_etc.plot_encircled_energy_mpl(
        image_clean=psf_img,
        pixel_scale_mas=imsim.plate_scale_mas,
        ax=ax_ee,
        units="mas",
        ee_target=None,
        label=name,
    )

ax_r.set_yscale("log")
ax_r.set_title("Azimuthally-averaged PSF")
ax_ee.set_title("Encircled energy")
ax_r.legend(fontsize=9)
ax_ee.legend(fontsize=9, loc="lower right")
fig.tight_layout()
plt.show()

## 3. Pointing jitter

Jitter is a Gaussian blur of the PSF: more jitter means a broader core and a lower peak
fraction, which is what drives the saturation budget in `02_saturation_flag.ipynb`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, jitter in zip(axes, [0.0, 20.0, 50.0]):
    img = imsim.render_psf(wcc_etc.AiryPSF(), jitter_sigma_mas=jitter, npix=96)
    im = ax.imshow(img, origin="lower", cmap="viridis")
    ax.set_title(f"jitter = {jitter:.0f} mas\npeak fraction = {img.max():.3f}")
    ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("Airy PSF vs pointing jitter (sony:r)", y=1.03)
fig.tight_layout()
plt.show()

## 4. Aperture size, defocus, and the optimum

`get_image_snr(time, psf=...)` computes an aperture SNR from the rendered PSF, and
`optimize=True` searches for the SNR-maximizing radius.

This is where defocus actually costs you. At the **default** 70 mas aperture — sized for the
diffraction-limited core — the +2 wave PSF puts most of its light outside the aperture and
the SNR collapses by roughly a factor of seven. Open the aperture to the optimum and most
of it comes back: the penalty is mostly a *mismatched aperture* rather than lost
photons. The price is a far larger aperture, which is much more exposed to crowding and
background.

Concretely, at 60 s: the +2 wave PSF gives SNR **40.6** at the default 69 mas aperture
against **306.7** in focus — a **7.6x** collapse. Opening up to the optimum 566 mas
recovers it to **273.4**, against **315.7** for the optimized in-focus case, so the
residual cost of two waves of defocus is about **13%**, not a factor of seven.

In [ ]:
radii = np.linspace(20, 800, 40)

fig, ax = plt.subplots(figsize=(7.6, 4.6))
for name, psf in PSFS.items():
    snr = [sim.get_image_snr(time=60, psf=psf, r_aper_mas=r)["snr"] for r in radii]
    (line,) = ax.plot(radii, snr, label=name)

    default = sim.get_image_snr(time=60, psf=psf)
    best = sim.get_image_snr(time=60, psf=psf, optimize=True)
    ax.plot(default["r_aper_mas"], default["snr"], "s", color=line.get_color(), ms=7)
    ax.plot(best["r_aper_mas"], best["snr"], "o", color=line.get_color(), ms=9)
    print(
        f"{name:<28}"
        f" default r={default['r_aper_mas']:6.1f} mas SNR={default['snr']:7.2f}"
        f"   optimized r={best['r_aper_mas']:6.1f} mas SNR={best['snr']:7.2f}"
    )

ax.set_xlabel("Aperture radius [mas]")
ax.set_ylabel("SNR (r = 18, 60 s)")
ax.set_title("Squares = default aperture, circles = optimize=True")
ax.legend(fontsize=9)
plt.show()

## 5. Bokeh backend

Every plot has a bokeh variant via `backend='bokeh'`. In a notebook call
`output_notebook()` once and `show(...)` the returned figure; for the web portal pass
`return_='components'` (script + div) or `return_='html'` instead.

In [ ]:
from bokeh.io import output_notebook, show

output_notebook()
show(res.plot_image(backend="bokeh", units="mas", title="Airy PSF (bokeh)"))

## Summary

- `AiryPSF`, `DefocusPSF`, and `CustomPSF` render normalized PSFs onto the detector grid,
  aware of pixel size and jitter.
- `ImageSimulator.render_psf(psf, jitter_sigma_mas=..., npix=...)` returns that bare PSF;
  `plate_scale_mas` converts pixels to mas for the plotting helpers.
- `ImageSimulator.simulate(time)` produces a realistic noisy electron image with a
  saturation mask; `to_adu()` / `to_fitsimg()` bridge to ADU and the photometry tools.
- `Simulation.get_image_snr` computes an aperture SNR from the rendered PSF; defocus costs
  SNR at a fixed aperture, and `optimize=True` finds the best radius.
- Every matplotlib plotter has a `backend='bokeh'` twin.